# Import Library

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import hstack
from collections import defaultdict
from difflib import get_close_matches
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import coo_matrix

# UI using tkinter
import sys
from PyQt5.QtWidgets import (QApplication, QMainWindow, QWidget, QVBoxLayout, 
                             QHBoxLayout, QLineEdit, QPushButton, QListWidget, 
                             QLabel, QStackedWidget, QComboBox, QSpinBox, 
                             QDoubleSpinBox, QMessageBox, QListWidgetItem)
from PyQt5.QtCore import Qt
from PyQt5.QtGui import QFont, QPixmap
from PyQt5.QtMultimedia import QMediaPlayer, QMediaContent
from PyQt5.QtCore import QUrl

# Content-Based Filtering

In [2]:
df = pd.read_csv('data/newMusicInfo.csv')
df.head()

,track_id,name,artist,spotify_preview_url,tags,genre,year,duration_ms,danceability,speechiness,tempo
0,TROUVHL128F426C441,Come as You Are,NIRVANA,https://p.scdn.co/mp3-preview/a1c11bb1cb231031...,"ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, GRUNGE",RNB,1991,218920,0.508,0.0400,120.012
1,TRLNZBD128F935E4D8,Creep,RADIOHEAD,https://p.scdn.co/mp3-preview/e7eb60e9466bc3a2...,"ROCK, ALTERNATIVE, INDIE, ALTERNATIVE_ROCK, IN...",RNB,2008,238640,0.515,0.0369,91.841
2,TRMZXEW128F9341FD5,The Scientist,COLDPLAY,https://p.scdn.co/mp3-preview/95cb9df1b056d759...,"ROCK, ALTERNATIVE, INDIE, POP, ALTERNATIVE_ROC...",ROCK,2007,311014,0.566,0.0242,146.365
3,TRIODZU128E078F3E2,Under the Bridge,RED HOT CHILI PEPPERS,https://p.scdn.co/mp3-preview/90e41778392f27b6...,"ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, FUNK",POP,2003,265506,0.554,0.0457,84.275
4,TRYEZGE128F92D04C8,Feel Good Inc.,GORILLAZ,https://p.scdn.co/mp3-preview/7388c425022ced92...,"ROCK, ELECTRONIC, ALTERNATIVE, INDIE, POP, ALT...",RNB,2010,223106,0.809,0.1360,138.556


### Selected only the relevant features

In [3]:
df_selected = df[['track_id', 'name','artist', 'tags', 'genre', 'year', 'danceability', 'speechiness', 'tempo','spotify_preview_url','duration_ms']]
df_selected.head()

,track_id,name,artist,tags,genre,year,danceability,speechiness,tempo,spotify_preview_url,duration_ms
0,TROUVHL128F426C441,Come as You Are,NIRVANA,"ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, GRUNGE",RNB,1991,0.508,0.0400,120.012,https://p.scdn.co/mp3-preview/a1c11bb1cb231031...,218920
1,TRLNZBD128F935E4D8,Creep,RADIOHEAD,"ROCK, ALTERNATIVE, INDIE, ALTERNATIVE_ROCK, IN...",RNB,2008,0.515,0.0369,91.841,https://p.scdn.co/mp3-preview/e7eb60e9466bc3a2...,238640
2,TRMZXEW128F9341FD5,The Scientist,COLDPLAY,"ROCK, ALTERNATIVE, INDIE, POP, ALTERNATIVE_ROC...",ROCK,2007,0.566,0.0242,146.365,https://p.scdn.co/mp3-preview/95cb9df1b056d759...,311014
3,TRIODZU128E078F3E2,Under the Bridge,RED HOT CHILI PEPPERS,"ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, FUNK",POP,2003,0.554,0.0457,84.275,https://p.scdn.co/mp3-preview/90e41778392f27b6...,265506
4,TRYEZGE128F92D04C8,Feel Good Inc.,GORILLAZ,"ROCK, ELECTRONIC, ALTERNATIVE, INDIE, POP, ALT...",RNB,2010,0.809,0.1360,138.556,https://p.scdn.co/mp3-preview/7388c425022ced92...,223106


### Check data type

In [4]:
df_selected.dtypes

track_id                object
name                    object
artist                  object
tags                    object
genre                   object
year                     int64
danceability           float64
speechiness            float64
tempo                  float64
spotify_preview_url     object
duration_ms              int64
dtype: object

### Copy and clean

In [5]:
df_selected = df[['track_id', 'name', 'artist', 'tags', 'genre', 'year', 'danceability', 'speechiness', 'tempo','spotify_preview_url','duration_ms']].copy()
df_selected[['name', 'artist', 'tags', 'genre']] = df_selected[['name', 'artist', 'tags', 'genre']].fillna('')

### Combine text features

In [6]:
df_selected['combined_text'] = df_selected['tags'] + ' ' + df_selected['genre'] + ' ' + df_selected['artist']

### TF-IDF on combined text

In [7]:
# Initialize TF-IDF vectorizer
tfidf = TfidfVectorizer(
    analyzer='word', 
    ngram_range=(1, 2),  # Include unigrams and bigrams
    min_df=1, # Keep words that appear in at least 1 document
    stop_words='english'
)

# Transform the combined text column into TF-IDF features
text_matrix = tfidf.fit_transform(df_selected['combined_text'])

### Normalize numeric features

In [8]:
# Initialize and apply MinMaxScaler to scale all numeric columns to [0, 1]
scaler = MinMaxScaler()
# Apply scaling to numerical features
numeric_matrix = scaler.fit_transform(df_selected[['danceability', 'speechiness', 'tempo', 'year']])

### Combine all features

In [9]:
combined_matrix = hstack([text_matrix, numeric_matrix]).tocsr()

### Fit KNN model

In [10]:
knn_model = NearestNeighbors(metric='cosine', algorithm='brute')
knn_model.fit(combined_matrix)

NearestNeighbors(algorithm='brute', metric='cosine')

### Build name, tag, genre, year and index mapping

In [11]:
# lowercase song names in mapping
# Build the (song name) -> index map
name_to_index = {
    row['name'].lower().strip(): idx
    for idx, row in df_selected.iterrows()
}

In [12]:
# Create a dictionary to map (tag, genre, year) -> list of row indices
tag_genre_year_to_indices = {}

# Loop through each row in the dataframe
for idx, row in df_selected.iterrows():
    # Build a unique key using tag, genre, and year (all lowercase for consistency)
    key = (row['tags'].lower().strip(), row['genre'].lower().strip(), row['year'])
    
    # Insert the key into the dictionary if it doesn't exist, and append the current row index to its list
    tag_genre_year_to_indices.setdefault(key, []).append(idx)

In [13]:
# Create a dictionary to map (genre, year) -> list of row indices
genre_year_to_indices = {}

# Loop through each row in the dataframe
for idx, row in df_selected.iterrows():
    # Build a unique key using genre and year (all lowercase for consistency)
    key = (row['genre'].lower().strip(), row['year'])
    
     # Insert the key into the dictionary if it doesn't exist, and append the current row index to its list
    genre_year_to_indices.setdefault(key, []).append(idx)

In [14]:
genre_to_indices = {}

# Loop through each row in the dataframe
for idx, row in df_selected.iterrows():
    # Build a unique key using genre (all lowercase for consistency)
    key = str(row['genre']).lower().strip()
    
     # Insert the key into the dictionary if it doesn't exist, and append the current row index to its list
    genre_to_indices.setdefault(key, []).append(idx)

### Recommendation function using name

##### convert time ms to min

In [15]:
def ms_to_minutes_seconds(ms):
    try:
        ms = int(ms)
        seconds = ms // 1000
        minutes = seconds // 60
        remaining_seconds = seconds % 60
        return f"{minutes}:{remaining_seconds:02d}"
    except Exception:
        return None

##### Calculation accuracy function

In [16]:
def calculate_accuracy_from_distance(distances):
    # Convert KNN distances into similarity percentages.
    return [(1 - d) * 100 for d in distances]

##### Validation for tempo, danceability and year

In [17]:
def validate_song_name(song_name):
    # Validate that the song exists in the dataset
    song_key = song_name.lower().strip()
    if song_key not in name_to_index:
        return None, f"Song '{song_name}' not found in the dataset."
    return song_key, None

##### Main Recommendation using name Function

In [18]:
def recommend_songs_by_name(song_name, n_recommendations=5):
    # Preprocess the input name (normalize: lowercase + remove spaces)
    song_key = song_name.lower().strip()
    
    # 1 - Check if the song exists in the mapping
    song_key, error = validate_song_name(song_name)
    if error:
        return error
    
    # 2 - Get the index of the input song directly from the dictionary
    idx = name_to_index[song_key] 
    
    # 3 - Find nearest neighbors using the KNN model
    distances, indices = knn_model.kneighbors(
        combined_matrix[idx], n_neighbors=n_recommendations + 1
    )
    
    # 4 - Exclude the first result (it’s always the song itself)
    recommended_indices = indices.flatten()[1:]
    recommended_distances = distances.flatten()[1:]
    
    # 5 - Get details of the input song
    input_song = df_selected.iloc[[idx]][
        ['track_id','name','artist','year','genre','tempo','tags','danceability','spotify_preview_url','duration_ms']
    ]
    
    # 6 - Get details of recommended songs
    recommendations = df_selected.iloc[recommended_indices][
        ['track_id','name','artist','year','genre','tempo','tags','danceability','spotify_preview_url','duration_ms']
    ]
       
    
    # 7 - Add similarity score ("accuracy") column and calculation 
    input_song = input_song.assign(accuracy=["100%"])
    similarities = calculate_accuracy_from_distance(recommended_distances)
    recommendations = recommendations.assign(
        accuracy=[f"{s:.2f}%" for s in similarities]
    )
    
    # 8 - Merge input + recommendations into one DataFrame
    result = pd.concat([input_song, recommendations]).reset_index(drop=True)
    
    # 9 - convert time ms to min
    result['duration'] = result['duration_ms'].apply(ms_to_minutes_seconds)
    result = result.drop(columns=['duration_ms'])
    
    return result

### Recommendation function using genre, tags, tempo and danceability

##### Calculation accuracy function

In [19]:
def calculate_accuracy(song_tempo, song_dance, input_tempo, input_dance):
    
    # Calculate similarity with input manually (tempo + danceability diff)
    tempo_diff = abs(song_tempo - input_tempo) / max(input_tempo, 1)
    dance_diff = abs(song_dance - input_dance)
    
    # Simple similarity formula: 1 - take 50% weight from tempo difference + 50% from danceability difference.
    sim = (1 - (0.5 * tempo_diff + 0.5 * dance_diff)) * 100
    
    # Clip the score between 0 and 100 to avoid negative or >100 values.
    sim = max(0, min(100, sim)) 
    
    return f"{sim:.2f}%"

##### Validation for tempo, danceability and year

In [20]:
def validate_inputs(song_tempo, danceability_value, year):
    
    # Validate tempo (must be numeric)
    try:
        song_tempo = float(song_tempo)
    except ValueError:
        return "Tempo must be a number (BPM)."

    # Validate danceability (0.0–1.0)
    try:
        danceability_value = float(danceability_value)
    except ValueError:
        return "Danceability must be between 0.0 and 1.0."
    if not (0.0 <= danceability_value <= 1.0):
        return "Danceability must be between 0.0 and 1.0."

    # Validate year (must be integer)
    try:
        year = int(year)
    except ValueError:
        return "Year must be an integer."

    return song_tempo, danceability_value, year


##### Main Recommendation using genre, tags, tempo and danceability Function

In [21]:
def recommend_songs_by_genre_tempo_tags_danceability(
    song_genre, song_tempo, song_tags, danceability_value, year, n_recommendations=5
):
    # Normalize text fields (remove spaces + lowercase for consistency)
    song_genre = song_genre.lower().strip()
    song_tags = song_tags.lower().strip()

    # Run input validation
    validated = validate_inputs(song_tempo, danceability_value, year)
    
    # error message returned
    if isinstance(validated, str):  
        return validated 
    song_tempo, danceability_value, year = validated

    # 1 - fallback search
    candidates = None
    key_tag = (song_tags, song_genre, year)
    
    # First try - (tag + genre + year) exact match
    if key_tag in tag_genre_year_to_indices:
        candidates = df_selected.iloc[tag_genre_year_to_indices[key_tag]]

    # Second try - (genre + year)
    elif (song_genre, year) in genre_year_to_indices:
        candidates = df_selected.iloc[genre_year_to_indices[(song_genre, year)]]

    # Third try - only genre
    elif song_genre in genre_to_indices:
        candidates = df_selected.iloc[genre_to_indices[song_genre]]

    # If nothing matches - return not found message
    if candidates is None or candidates.empty:
        return f"No songs found in genre '{song_genre}' for year {year} (with/without tag '{song_tags}')."

    # 2 - Tempo filter ±5
    tempo_matches = candidates[(candidates['tempo'] >= song_tempo - 5) & (candidates['tempo'] <= song_tempo + 5)]
    if tempo_matches.empty:
        return f"No songs in genre '{song_genre}' near {song_tempo} BPM."

    # 3 - Danceability filter ±0.1
    final_matches = tempo_matches[
        (tempo_matches['danceability'] >= danceability_value - 0.1) & 
        (tempo_matches['danceability'] <= danceability_value + 0.1)
    ].copy()
    if final_matches.empty:
        return f"No songs in genre '{song_genre}' near {song_tempo} BPM with danceability {danceability_value}."

    # 4 - Pick the "closest" candidate (min absolute difference in danceability)
    final_matches['diff'] = (
        (final_matches['tempo'] - song_tempo).abs() +
        (final_matches['danceability'] - danceability_value).abs() + 
        (final_matches['year'] - year).abs()
    )
    idx = final_matches['diff'].idxmin()

    # 5 - Run KNN
    distances,indices = knn_model.kneighbors(combined_matrix[idx], n_neighbors=n_recommendations + 1)
    recommended_indices = indices.flatten()[1:]
    # recommended_distances = distances.flatten()[1:]

    # 6 - Get details of the input song
    input_song = df_selected.iloc[[idx]][['track_id','name','artist','year','genre','tempo','tags','danceability','spotify_preview_url','duration_ms']]
    
    # Check if input exactly matches the chosen candidate (strict match on 4 fields)
    is_exact_match = (
        input_song['genre'].iloc[0].lower().strip() == song_genre and
        input_song['tags'].iloc[0].lower().strip() == song_tags and
        input_song['year'].iloc[0] == year and
        abs(input_song['tempo'].iloc[0] - song_tempo) < 1e-6 and
        abs(input_song['danceability'].iloc[0] - danceability_value) < 1e-6
    )
    
    # 7 - Get details of recommended songs
    recommendations = df_selected.iloc[recommended_indices][['track_id','name','artist','year','genre','tempo','tags','danceability','spotify_preview_url','duration_ms']]

    # 8 - Add similarity score ("accuracy") column and calculation 
    if is_exact_match:
        input_song = input_song.assign(accuracy=["100%"])
    else:
        ## Input song accuracy     
        acc = calculate_accuracy(
            input_song['tempo'].iloc[0],
            input_song['danceability'].iloc[0],
            song_tempo,
            danceability_value
        )
        input_song = input_song.assign(accuracy=[acc])
        
    ## Recommended songs accuracy
    # For each recommended song, calculate similarity to the user’s input values.
    recommendations = df_selected.iloc[recommended_indices][['track_id','name','artist','year','genre','tempo','tags','danceability','spotify_preview_url','duration_ms']].copy()
    recommendations['accuracy_val'] = recommendations.apply(
        lambda row: float(calculate_accuracy(row['tempo'], row['danceability'], song_tempo, danceability_value).replace("%","")),
        axis=1
    )
    
    # Sort by accuracy high → low
    recommendations = recommendations.sort_values(by="accuracy_val", ascending=False)
    
    # Format accuracy
    recommendations['accuracy'] = recommendations['accuracy_val'].map(lambda s: f"{s:.2f}%")
    recommendations = recommendations.drop(columns="accuracy_val")

    # 9 - Merge input + recommendations into one DataFrame
    result =  pd.concat([input_song, recommendations]).reset_index(drop=True)
    
    # 10 - convert time ms to min
    result['duration'] = result['duration_ms'].apply(ms_to_minutes_seconds)
    result = result.drop(columns=['duration_ms'])
    
    return result


# Collaborative Filtering System

## Then load the datasets 

In [22]:
user_history = pd.read_csv('data/newUserListeningHistory.csv')
music_info = pd.read_csv('data/newMusicInfo.csv')

## encode user and track id

In [23]:
user_encoder = LabelEncoder()
track_encoder = LabelEncoder()
user_history['user_index'] = user_encoder.fit_transform(user_history['user_id'])
user_history['track_index'] = track_encoder.fit_transform(user_history['track_id'])

## create sparse item matrix

In [24]:
rows = user_history['user_index']
cols = user_history['track_index']
data = user_history['playcount'].astype(np.float32)
num_users = rows.nunique()
num_items = cols.nunique()
user_item_matrix = coo_matrix((data, (rows, cols)), shape=(num_users, num_items)).tocsr()

## apply truncated SVD

In [25]:
svd = TruncatedSVD(n_components=20, random_state=42)
svd_matrix = svd.fit_transform(user_item_matrix)

## convert index to id to map them

In [26]:
track_id_to_info = music_info.set_index('track_id')[['name', 'artist']].to_dict('index')
user_index_to_id = dict(enumerate(user_encoder.classes_))
track_index_to_id = dict(enumerate(track_encoder.classes_))

## recommend based on user ID then compare for similarity then suggest song that havent exist in the album of the user input id

In [27]:
def recommend_by_user_id(user_id_input, top_n=5):
    if user_id_input not in user_encoder.classes_:
        suggestions = get_close_matches(user_id_input, user_encoder.classes_, n=3)
        return f"User ID not found. Did you mean: {', '.join(suggestions)}?"

    user_index = user_encoder.transform([user_id_input])[0]
    target_vector = svd_matrix[user_index]
    
    # Calculate cosine similarity with ALL users
    similarities = cosine_similarity([target_vector], svd_matrix)[0]

    # Get top 5 similar users (excluding self)
    top_user_indices = np.argsort(-similarities)[1:6]
    
    print(f"\nInput User ID: {user_id_input}")
    print("Top 5 similar users and their similarity scores:")
    user_similarities = {}
    for i, user_idx in enumerate(top_user_indices, 1):
        sim_id = user_index_to_id[user_idx]
        sim_score = similarities[user_idx]
        user_similarities[user_idx] = sim_score
        print(f" {i}. {sim_id} (Similarity: {sim_score:.4f})")

    print(f"\nAnalyzing songs from these {len(top_user_indices)} similar users...")
    
    # Track which users listened to which songs
    track_to_user_similarities = defaultdict(list)
    
    for sim_idx in top_user_indices:
        uid = user_index_to_id[sim_idx]
        sim_score = similarities[sim_idx]
        user_tracks = user_history[user_history['user_index'] == sim_idx]['track_index'].unique()
        
        for tidx in user_tracks:
            track_to_user_similarities[tidx].append({
                'user_id': uid,
                'user_index': sim_idx,
                'similarity': sim_score
            })

    # Get songs user has already listened to
    listened = set(user_history[user_history['user_index'] == user_index]['track_index'].unique())
    
    # Score tracks based on weighted average of similar users' preferences
    track_scores = {}
    for track_idx, user_list in track_to_user_similarities.items():
        if track_idx not in listened:  # Only recommend unheard songs
            # Weight by similarity scores
            weighted_score = sum(user_info['similarity'] for user_info in user_list)
            track_scores[track_idx] = {
                'score': weighted_score,
                'contributing_users': user_list
            }
    
    # Sort by weighted score and get top N
    top_tracks = sorted(track_scores.items(), key=lambda x: x[1]['score'], reverse=True)[:top_n]
    
    recommendations = []
    for track_idx, track_data in top_tracks:
        track_id = track_index_to_id[track_idx]
        info = track_id_to_info.get(track_id, {'name': 'Unknown', 'artist': 'Unknown'})
        
        # Format contributing users with their similarities
        contributing_users = []
        for user_info in track_data['contributing_users']:
            contributing_users.append(f"{user_info['user_id']} ({user_info['similarity']:.4f})")

         # Get Spotify URL and duration from original data
        track_row = df[df['track_id'] == track_id]
        spotify_url = track_row['spotify_preview_url'].values[0] if not track_row.empty else 'Unknown'
        duration_ms = track_row['duration_ms'].values[0] if not track_row.empty else 0
        
        # Convert duration from ms to mm:ss format
        duration_formatted = f"{int(duration_ms)//60000}:{(int(duration_ms)%60000)//1000:02d}" if duration_ms else '0:00'
        
        recommendations.append({
            'track_id': track_id,
            'name': info['name'],
            'artist': info['artist'],
            'weighted_score': track_data['score'],
            'num_similar_users': len(track_data['contributing_users']),
            'contributing_users_with_similarity': '; '.join(contributing_users),  
            'spotify_preview_url': spotify_url,  
            'duration': duration_formatted  
        })
    
    return pd.DataFrame(recommendations)

# Hybrid Recommendation System

In [28]:
# Hybrid recommendation function (combines content-based and collaborative filtering)
def hybrid_recommendation(song_name, user_id, n_recommendations=5):
    """
    Hybrid recommendation that combines:
    1. Content-based filtering based on the searched song
    2. Collaborative filtering based on user preferences
    """
    # Get content-based recommendations
    content_recs = recommend_songs_by_name(song_name, n_recommendations * 2)
    
    # Get collaborative recommendations
    collab_recs = recommend_by_user_id(user_id, n_recommendations * 2)
    
    # Combine and deduplicate (simple hybrid approach)
    # In a real implementation, you would use a more sophisticated blending method
    hybrid_results = pd.concat([content_recs, collab_recs]).drop_duplicates(subset=['track_id'])
    
    # Return top n recommendations
    return hybrid_results.head(n_recommendations)

# UI and Hybrid Recommendation System

In [29]:
# ===========================
# song-style UI (GLUE ONLY)
# - Uses your existing functions EXACTLY as defined earlier:
#     recommend_songs_by_name(song_name, n_recommendations)
#     recommend_by_user_id(user_id_input, top_n)
#     (optional) hybrid_recommendation(song_name, user_id, n_recommendations)
# - No algorithm changes here. This is just UI + helpers.
# ===========================

from __future__ import annotations
import sys
from typing import List, Dict, Any, Optional

import numpy as np
import pandas as pd

from PyQt5.QtCore import Qt, QUrl
from PyQt5.QtGui import QFont, QPixmap
from PyQt5.QtWidgets import (
    QApplication, QMainWindow, QWidget, QVBoxLayout, QHBoxLayout, QGridLayout,
    QLabel, QLineEdit, QPushButton, QToolButton, QFrame, QScrollArea,
    QStackedWidget, QSplitter, QSlider, QDialog, QFormLayout,
    QDialogButtonBox, QSizePolicy
)
from PyQt5.QtMultimedia import QMediaPlayer, QMediaContent

# ---------- Theme tokens ----------
SONG_GREEN = "#1DB954"
BG = "#121212"; BG_ELEV_1 = "#181818"; BG_ELEV_2 = "#242424"
TEXT_PRIMARY = "#FFFFFF"; TEXT_SECONDARY = "#B3B3B3"; BORDER = "#282828"

APP_QSS = f"""
QMainWindow, QWidget {{ background-color: {BG}; color: {TEXT_PRIMARY}; }}
QLabel {{ color: {TEXT_PRIMARY}; }}
QLineEdit {{ background: {BG_ELEV_2}; color: {TEXT_PRIMARY}; border: 1px solid {BORDER}; border-radius: 16px; padding: 8px 12px; }}
QToolButton, QPushButton {{ background: transparent; color: {TEXT_PRIMARY}; border: none; padding: 6px 10px; }}
QToolButton:hover, QPushButton:hover {{ color: {TEXT_PRIMARY}; background: {BG_ELEV_2}; border-radius: 8px; }}
QPushButton#primary {{ background: {SONG_GREEN}; color: #000; border-radius: 20px; padding: 8px 16px; font-weight: 600; }}
QFrame#sidebar {{ background: #000000; border-right: 1px solid {BORDER}; }}
QFrame#topbar {{ background: {BG}; border-bottom: 1px solid {BORDER}; }}
QFrame#card {{ background: {BG_ELEV_1}; border: 1px solid {BORDER}; border-radius: 8px; }}
QScrollArea {{ border: none; }}
QSlider::groove:horizontal {{ height: 4px; background: {BORDER}; border-radius: 2px; }}
QSlider::handle:horizontal {{ width: 12px; height: 12px; margin: -4px 0; background: {SONG_GREEN}; border-radius: 6px; }}
QPushButton#playpause, QToolButton#playpause {{ background: transparent; color: {TEXT_PRIMARY}; border: 1px solid {BORDER}; border-radius: 12px; padding: 4px 8px; }}
QPushButton#playpause:hover, QToolButton#playpause:hover {{ background: {BG_ELEV_2}; }}
QPushButton#playpause:pressed, QToolButton#playpause:pressed {{ background: {BG_ELEV_2}; }}
QPushButton#playpause:checked, QToolButton#playpause:checked {{ background: {BG_ELEV_2}; }}
"""

# ---------- ADAPTERS (call your existing functions; zero reimplementation) ----------
def recs_content(query: str, k: int):
    # defined earlier in your notebook
    return recommend_songs_by_name(query, n_recommendations=k)

def recs_collab(user_id: str, k: int):
    # defined earlier in your notebook
    return recommend_by_user_id(user_id, top_n=k)

# ---------- Utility ----------
def ms_to_mmss(ms: Optional[float]) -> str:
    if ms is None or (isinstance(ms, float) and np.isnan(ms)): return "0:00"
    try:
        s = int(ms) // 1000; m, s = divmod(s, 60); return f"{m}:{s:02d}"
    except Exception:
        return "0:00"

def _pick(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    for c in candidates:
        if c in df.columns: return c
    return None

def _to_cards(df: pd.DataFrame) -> List[Dict[str, Any]]:
    """Tolerant normalizer → UI card dicts."""
    if df is None or not isinstance(df, pd.DataFrame) or df.empty: return []
    title  = _pick(df, ["name","title","track_name","song","track"]) or df.columns[0]
    artist = _pick(df, ["artist","artists","artist_name","primary_artist"])
    cover  = _pick(df, ["image_url","album_image_url","cover_url","album_cover_url","thumbnail"])
    prev   = _pick(df, ["spotify_preview_url","preview_url","audio_preview_url"])
    durms  = _pick(df, ["duration_ms","length_ms","length"])
    dur    = _pick(df, ["duration","duration_str"])

    cards = []
    for _, r in df.iterrows():
        cards.append({
            "track_id": r.get("track_id",""),
            "name": str(r.get(title, "")),
            "artist": str(r.get(artist,"")) if artist else "",
            "spotify_preview_url": r.get(prev,"") if prev else "",
            "duration": ms_to_mmss(r.get(durms)) if durms else (str(r.get(dur,"")) if dur else ""),
            "album_art_url": r.get(cover, None)
        })
    return cards

# ---------- Ensure data exists for Trending/New (no algorithm changes) ----------
def _ensure_trending_sources_loaded():
    """
    Ensure _music_info and _user_history exist.
    1) If your notebook has build_collab_models(), call it.
    2) Else, read CSVs directly from data/.
    """
    import pandas as _pd  # local alias to avoid shadowing

    def _missing(name: str) -> bool:
        try:
            obj = globals()[name]
            return not isinstance(obj, _pd.DataFrame) or obj.empty
        except Exception:
            return True

    need_music = _missing("_music_info")
    need_hist  = _missing("_user_history")
    if not (need_music or need_hist):
        return

    # Try user's builder first (no code change to algorithms)
    if "build_collab_models" in globals() and callable(globals()["build_collab_models"]):
        try:
            build_collab_models()   # user-defined
            return
        except Exception:
            pass  # fall back to CSVs

    # Fallback: CSVs
    try:
        if need_music:
            globals()["_music_info"] = _pd.read_csv("data/newMusicInfo.csv")
    except Exception:
        globals()["_music_info"] = _pd.DataFrame()
    try:
        if need_hist:
            globals()["_user_history"] = _pd.read_csv("data/newUserListeningHistory.csv")
    except Exception:
        globals()["_user_history"] = _pd.DataFrame()

# ---------- ONE function for both 'Trending now' and 'New Releases' ----------
def compute_trending_and_new(n_trending: int = 20, n_new: int = 20):
    """
    Returns (trending_cards, new_release_cards) using your original logic:
      - Trending: aggregate _user_history by playcount and map to track info
      - New releases: sort _music_info by 'year' (or 'release_date') desc
    It auto-loads data if missing (without changing algorithms).
    """
    _ensure_trending_sources_loaded()

    trending = []
    # --- Trending ---
    try:
        dfh = globals().get("_user_history", None)
        if isinstance(dfh, pd.DataFrame) and not dfh.empty:
            if "track_id" in dfh.columns:
                if "playcount" in dfh.columns:
                    agg = (dfh.groupby("track_id")["playcount"].sum()
                           .sort_values(ascending=False).head(n_trending))
                else:
                    agg = (dfh.groupby("track_id").size()
                           .sort_values(ascending=False).head(n_trending))
                track_ids = list(agg.index)
            elif "track_index" in dfh.columns and "_track_index_to_id" in globals():
                if "playcount" in dfh.columns:
                    agg = (dfh.groupby("track_index")["playcount"].sum()
                           .sort_values(ascending=False).head(n_trending))
                else:
                    agg = (dfh.groupby("track_index").size()
                           .sort_values(ascending=False).head(n_trending))
                track_ids = [ globals()["_track_index_to_id"].get(int(i)) for i in agg.index.tolist() ]
            else:
                track_ids = []

            if track_ids:
                # Prefer user's prebuilt lookup dict
                if "_track_id_to_info" in globals() and globals()["_track_id_to_info"]:
                    info_map = globals()["_track_id_to_info"]
                    for tid in track_ids:
                        info = info_map.get(tid, {})
                        trending.append({
                            "track_id": tid,
                            "name": info.get("name", ""),
                            "artist": info.get("artist", ""),
                            "spotify_preview_url": info.get("spotify_preview_url", ""),
                            "duration": ms_to_mmss(info.get("duration_ms")),
                            "album_art_url": info.get("image_url", None),
                        })
                else:
                    mi = globals().get("_music_info", None)
                    if isinstance(mi, pd.DataFrame) and "track_id" in mi.columns:
                        sub = mi[mi["track_id"].isin(track_ids)]
                        trending = _to_cards(sub)
    except Exception:
        pass

    # Fallbacks for trending if empty
    mi = globals().get("_music_info", None)
    if not trending and isinstance(mi, pd.DataFrame) and not mi.empty:
        tmp = mi.copy()
        for c in ["popularity", "playcount", "streams", "views"]:
            if c in tmp.columns:
                trending = _to_cards(tmp.sort_values(c, ascending=False).head(n_trending))
                break
        if not trending:
            trending = _to_cards(tmp.head(n_trending))

    # --- New Releases ---
    new_releases = []
    try:
        if isinstance(mi, pd.DataFrame) and not mi.empty:
            df = mi.copy()
            if "year" in df.columns:
                df["year"] = pd.to_numeric(df["year"], errors="coerce")
                df = df.sort_values("year", ascending=False).head(n_new)
                new_releases = _to_cards(df)
            elif "release_date" in df.columns:
                df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
                df = df.sort_values("release_date", ascending=False).head(n_new)
                new_releases = _to_cards(df)
            else:
                new_releases = _to_cards(df.head(n_new))
    except Exception:
        pass

    return trending, new_releases

# ---------- UI widgets ----------
class HScrollArea(QScrollArea):
    def __init__(self):
        super().__init__()
        self.setWidgetResizable(True)
        self.setFrameShape(QFrame.NoFrame)
        self.setHorizontalScrollBarPolicy(Qt.ScrollBarAlwaysOff)
        self.setVerticalScrollBarPolicy(Qt.ScrollBarAlwaysOff)
        self.setStyleSheet("QScrollArea { background: transparent; }")

class LoginDialog(QDialog):
    def __init__(self, on_login):
        super().__init__()
        self.setWindowTitle("Log in")
        self.setModal(True)
        self.on_login = on_login
        form = QFormLayout()
        self.edit_user = QLineEdit(); self.edit_user.setPlaceholderText("Enter your User ID")
        form.addRow("User ID", self.edit_user)
        btns = QDialogButtonBox(QDialogButtonBox.Ok | QDialogButtonBox.Cancel)
        btns.accepted.connect(self.try_login); btns.rejected.connect(self.reject)
        v = QVBoxLayout(self); v.addLayout(form); v.addWidget(btns)

    def try_login(self):
        u = self.edit_user.text().strip()
        if not u: return
        ok = self.on_login(u) if callable(self.on_login) else True
        if ok: self.accept()

class TrackCard(QFrame):
    def __init__(self, track: dict, on_play):
        super().__init__()
        self.setObjectName("card")
        self.track = track; self.on_play = on_play
        self.setSizePolicy(QSizePolicy.Fixed, QSizePolicy.Fixed)
        self.setFixedWidth(140); self.setFixedHeight(220)

        v = QVBoxLayout(self); v.setContentsMargins(10,10,10,10); v.setSpacing(6)
        cover = QLabel(); cover.setFixedSize(110,110)
        art = track.get("album_art_url")
        if art:
            pix = QPixmap(str(art))
            if not pix.isNull():
                cover.setPixmap(pix.scaled(110,110, Qt.KeepAspectRatioByExpanding, Qt.SmoothTransformation))
        if cover.pixmap() is None:
            letter = (track.get("name") or "?")[:1].upper()
            cover.setAlignment(Qt.AlignCenter)
            cover.setStyleSheet(f"background:{BG_ELEV_2};border-radius:6px; color:{TEXT_SECONDARY};")
            cover.setFont(QFont("Arial", 28, QFont.Bold)); cover.setText(letter)

        title = QLabel(track.get("name") or "Unknown"); title.setWordWrap(True); title.setFont(QFont("Arial", 9, QFont.DemiBold))
        artist = QLabel(track.get("artist") or ""); artist.setStyleSheet(f"color:{TEXT_SECONDARY}"); artist.setWordWrap(True); artist.setFont(QFont("Arial", 8))
        btn = QToolButton(); btn.setText("▶ Play"); btn.clicked.connect(self.play)

        v.addWidget(cover); v.addWidget(title); v.addWidget(artist); v.addStretch(1); v.addWidget(btn)

    def play(self):
        if callable(self.on_play): self.on_play(self.track)

class SectionGrid(QWidget):
    """Horizontal row or vertical grid of cards."""
    def __init__(self, title: str, on_play, min_height: int = 240, empty_text: str = "No items.", orientation: str = "horizontal"):
        super().__init__()
        self.on_play = on_play; self.empty_text = empty_text; self.orientation = orientation
        v = QVBoxLayout(self); self.lbl = QLabel(title); self.lbl.setFont(QFont("Arial", 16, QFont.Bold))
        v.addWidget(self.lbl)

        if orientation == "horizontal":
            self.container = QWidget(); self.h = QHBoxLayout(self.container)
            self.h.setContentsMargins(4,4,4,4); self.h.setSpacing(16); self.h.setAlignment(Qt.AlignLeft|Qt.AlignTop)
            self.scroll = HScrollArea(); self.scroll.setWidget(self.container); self.scroll.setFixedHeight(min_height + 28)
            v.addWidget(self.scroll)
        else:
            self.container = QWidget(); self.grid = QGridLayout(self.container)
            self.grid.setContentsMargins(4,4,4,4); self.grid.setHorizontalSpacing(16); self.grid.setVerticalSpacing(16)
            self.scroll = QScrollArea(); self.scroll.setWidgetResizable(True); self.scroll.setFrameShape(QFrame.NoFrame)
            self.scroll.setHorizontalScrollBarPolicy(Qt.ScrollBarAlwaysOff); self.scroll.setVerticalScrollBarPolicy(Qt.ScrollBarAsNeeded)
            self.scroll.setStyleSheet("QScrollArea { background: transparent; }"); self.scroll.setWidget(self.container)
            v.addWidget(self.scroll)

    def set_title(self, title: str): self.lbl.setText(title)

    def _clear(self):
        for child in self.container.findChildren(QWidget):
            child.setParent(None)

    def render_tracks(self, tracks: List[Dict[str, Any]]):
        self._clear()
        if not tracks:
            empty = QLabel(self.empty_text); empty.setStyleSheet(f"color:{TEXT_SECONDARY}")
            if self.orientation == "horizontal":
                self.h.addWidget(empty); self.h.addStretch(1)
            else:
                self.grid.addWidget(empty, 0, 0, 1, 5)
            return
        if self.orientation == "horizontal":
            for t in tracks: self.h.addWidget(TrackCard(t, self.on_play))
            self.h.addStretch(1)
        else:
            cols = 5
            for i, t in enumerate(tracks):
                r, c = divmod(i, cols)
                self.grid.addWidget(TrackCard(t, self.on_play), r, c)

class HomePage(QWidget):
    def __init__(self, on_play):
        super().__init__()
        outer = QVBoxLayout(self); outer.setContentsMargins(0,0,0,0)
        self.vscroll = QScrollArea(); self.vscroll.setWidgetResizable(True); self.vscroll.setFrameShape(QFrame.NoFrame)
        self.vscroll.setHorizontalScrollBarPolicy(Qt.ScrollBarAlwaysOff); self.vscroll.setVerticalScrollBarPolicy(Qt.ScrollBarAsNeeded)
        content = QWidget(); v = QVBoxLayout(content); v.setContentsMargins(12,8,12,8)
        self.sec_recs  = SectionGrid("Recommendations", on_play, min_height=240, empty_text="Log in to see personalized picks.", orientation="horizontal")
        self.sec_trend = SectionGrid("Trending now",   on_play, min_height=240, empty_text="No trending data yet.", orientation="horizontal")
        self.sec_new   = SectionGrid("New Releases",   on_play, min_height=240, empty_text="No recent releases found.", orientation="horizontal")
        v.addWidget(self.sec_recs); v.addWidget(self.sec_trend); v.addWidget(self.sec_new); v.addStretch(1)
        self.vscroll.setWidget(content); outer.addWidget(self.vscroll)
        self.set_recommendations_visible(False)

    def set_data(self, recs, trending, newrel):
        self.sec_recs.render_tracks(recs)
        self.sec_trend.render_tracks(trending)
        self.sec_new.render_tracks(newrel)

    def set_recommendations_visible(self, visible: bool):
        self.sec_recs.setVisible(visible)

class Sidebar(QFrame):
    def __init__(self):
        super().__init__(); self.setObjectName("sidebar"); self.setFixedWidth(240)
        v = QVBoxLayout(self); v.setContentsMargins(16,16,16,16)
        logo = QLabel("song"); logo.setFont(QFont("Arial", 18, QFont.Black))
        v.addWidget(logo); v.addSpacing(12)
        self.btn_home = QToolButton(); self.btn_home.setText("Home"); self.btn_home.setToolButtonStyle(Qt.ToolButtonTextOnly)
        self.btn_search = QToolButton(); self.btn_search.setText("Search"); self.btn_search.setToolButtonStyle(Qt.ToolButtonTextOnly)
        self.btn_library = QToolButton(); self.btn_library.setText("Your Library"); self.btn_library.setToolButtonStyle(Qt.ToolButtonTextOnly)
        for b in (self.btn_home, self.btn_search, self.btn_library):
            b.setCursor(Qt.PointingHandCursor); v.addWidget(b)
        v.addStretch(1)
        self.btn_login = QPushButton("Log in"); self.btn_login.setObjectName("primary"); v.addWidget(self.btn_login)

class TopBar(QFrame):
    def __init__(self):
        super().__init__(); self.setObjectName("topbar")
        h = QHBoxLayout(self); h.setContentsMargins(16,10,16,10)
        self.search = QLineEdit(); self.search.setPlaceholderText("What do you want to play? (search = content-based)")
        self.btn_go = QPushButton("Search")
        h.addWidget(self.search, 3); h.addWidget(self.btn_go); h.addStretch(1)
        self.user_label = QLabel(""); self.user_label.setStyleSheet(f"color:{TEXT_SECONDARY}"); h.addWidget(self.user_label)
        self.on_search = None
        self.search.returnPressed.connect(self.emit_search); self.btn_go.clicked.connect(self.emit_search)

    def emit_search(self):
        if callable(self.on_search):
            q = self.search.text().strip()
            if q: self.on_search(q)

class NowPlayingBar(QFrame):
    def __init__(self):
        super().__init__(); self.setStyleSheet(f"background:{BG_ELEV_1}; border-top:1px solid {BORDER};")
        v = QHBoxLayout(self); v.setContentsMargins(16,8,16,8)
        self.lbl_title = QLabel("Not playing"); self.lbl_artist = QLabel(""); self.lbl_artist.setStyleSheet(f"color:{TEXT_SECONDARY}")
        left = QVBoxLayout(); left.addWidget(self.lbl_title); left.addWidget(self.lbl_artist)
        self.btn_playpause = QPushButton(); self.btn_playpause.setText("▶"); self.btn_playpause.setCursor(Qt.PointingHandCursor)
        self.btn_playpause.setObjectName("playpause"); self.btn_playpause.setFlat(True); self.btn_playpause.setFocusPolicy(Qt.NoFocus)
        self.slider = QSlider(Qt.Horizontal); self.slider.setRange(0,1000)
        v.addLayout(left,2); v.addWidget(self.btn_playpause); v.addWidget(self.slider,5)

# ---------- Main Window ----------
class SpotifyMainWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("song Player"); self.setMinimumSize(1200, 820)
        self.current_user: str = ""

        # Player
        self.player = QMediaPlayer()
        self.player.positionChanged.connect(self._on_pos)
        self.player.durationChanged.connect(self._on_dur)

        # Layout
        central = QWidget(); outer = QVBoxLayout(central); outer.setContentsMargins(0,0,0,0); outer.setSpacing(0)
        self.topbar = TopBar(); self.topbar.on_search = self.do_search
        split = QSplitter(); self.sidebar = Sidebar()
        self.sidebar.btn_home.clicked.connect(self.show_home)
        self.sidebar.btn_search.clicked.connect(self.show_search)
        self.sidebar.btn_login.clicked.connect(self.toggle_auth)

        self.pages = QStackedWidget()
        self.page_home = HomePage(self.play_track)
        self.page_search = SectionGrid("Search results", self.play_track, min_height=260, orientation="vertical")
        self.pages.addWidget(self.page_home); self.pages.addWidget(self.page_search)

        split.addWidget(self.sidebar); split.addWidget(self.pages); split.setSizes([240,960])
        self.now = NowPlayingBar(); self.now.btn_playpause.clicked.connect(self.toggle_play)

        outer.addWidget(self.topbar); outer.addWidget(split,1); outer.addWidget(self.now)
        self.setCentralWidget(central)

        self.setStyleSheet(APP_QSS)  # avoid blue accent etc.
        self.populate_home(); self.update_auth_ui()

    # ----- Pages -----
    def populate_home(self):
        show_recs = bool(self.current_user)
        self.page_home.set_recommendations_visible(show_recs)

        recs = get_recommendations_for_user(self.current_user, n=20) if show_recs else []
        trending, newrel = compute_trending_and_new(n_trending=20, n_new=20)

        self.page_home.set_data(recs, trending, newrel)

    def show_home(self): self.pages.setCurrentWidget(self.page_home)
    def show_search(self): self.pages.setCurrentWidget(self.page_search)

    def show_login(self):
        def on_login(u: str):
            self.current_user = u
            self.populate_home(); self.update_auth_ui()
            return True
        LoginDialog(on_login).exec_()

    def update_auth_ui(self):
        self.sidebar.btn_login.setText("Log out" if self.current_user else "Log in")
        self.topbar.user_label.setText(self.current_user or "")
        self.page_home.set_recommendations_visible(bool(self.current_user))

    def toggle_auth(self):
        if self.current_user: self.logout()
        else: self.show_login()

    def logout(self):
        self.current_user = ""
        self.update_auth_ui(); self.populate_home()

    # ----- Search (CONTENT-BASED ONLY) -----
    def do_search(self, query: str):
        df = recs_content(query, k=30)
        if isinstance(df, str):  # e.g., "Song 'x' not found"
            tracks = []; self.page_search.set_title(f"No matches for: {query}")
        else:
            tracks = _to_cards(df); self.page_search.set_title(f"Search results for: {query}")
        self.page_search.render_tracks(tracks)
        self.show_search()

    # ----- Playback -----
    def play_track(self, track: dict):
        url = track.get('spotify_preview_url') or track.get('preview_url') or ''
        if not url: return
        self.player.setMedia(QMediaContent(QUrl(url)))
        self.player.play()
        self.now.lbl_title.setText(track.get('name') or '')
        self.now.lbl_artist.setText(track.get('artist') or '')
        self.now.btn_playpause.setText("⏸")

    def toggle_play(self):
        st = self.player.state()
        if st == QMediaPlayer.PlayingState:
            self.player.pause(); self.now.btn_playpause.setText("▶")
        else:
            self.player.play(); self.now.btn_playpause.setText("⏸")

    def _on_pos(self, pos):
        dur = max(1, self.player.duration())
        self.now.slider.blockSignals(True)
        self.now.slider.setValue(int(pos / dur * 1000))
        self.now.slider.blockSignals(False)

    def _on_dur(self, dur): self.now.slider.setValue(0)

# ----- Small helper to fetch user recs via collaborative -----
def get_recommendations_for_user(user_id: str, n: int = 20) -> List[Dict[str, Any]]:
    if not user_id: return []
    df = recs_collab(user_id, n)
    return _to_cards(df) if isinstance(df, pd.DataFrame) else []

# ----- Run app -----
if __name__ == "__main__":
    app = QApplication(sys.argv)
    w = SpotifyMainWindow()
    w.show()
    app.setStyleSheet(APP_QSS)
    sys.exit(app.exec_())


SystemExit: 0

C:\Users\Lwin\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
